In [0]:
-- 1. Create or Replace the Main Gold Volatility Leaderboard with Advanced Metrics
CREATE OR REPLACE TABLE crypto_gold_volatility AS
SELECT 
    coin_id,
    symbol,
    name,
    current_price,
    market_cap,
    market_cap_tier,
    price_spread_24h,
    price_change_percentage_24h,
    is_highly_volatile,
    
    -- Custom Volatility Index calculation
    ROUND(((high_24h - low_24h) / current_price) * 100, 2) AS volatility_index,
    
    -- ADVANCED ANALYTICS 1: Market Dominance % (Window Function)
    ROUND((market_cap / SUM(market_cap) OVER()) * 100, 2) AS market_dominance_pct,
    
    -- ADVANCED ANALYTICS 2: Tier Ranking
    DENSE_RANK() OVER (PARTITION BY market_cap_tier ORDER BY ((high_24h - low_24h) / current_price) DESC) as volatility_rank_in_tier,
    
    last_updated_timestamp
FROM crypto_silver;


-- 2. Create a Secondary Gold Table: Executive Market Summary by Tier
CREATE OR REPLACE TABLE crypto_gold_tier_summary AS
SELECT 
    market_cap_tier,
    COUNT(coin_id) AS total_coins_tracked,
    ROUND(AVG(current_price), 2) AS avg_price,
    ROUND(AVG(volatility_index), 2) AS avg_volatility_index,
    ROUND(SUM(market_cap), 2) AS total_tier_market_cap,
    SUM(CASE WHEN is_highly_volatile = true THEN 1 ELSE 0 END) AS highly_volatile_count
FROM crypto_gold_volatility
GROUP BY market_cap_tier
ORDER BY total_tier_market_cap DESC;


-- 3. DISPLAY THE RESULTS (This forces Databricks to show rows)
SELECT '--- MAIN VOLATILITY LEADERBOARD ---' as section;
SELECT * FROM crypto_gold_volatility LIMIT 10;

SELECT '--- TIER SUMMARY REPORT ---' as section;
SELECT * FROM crypto_gold_tier_summary;

market_cap_tier,total_coins_tracked,avg_price,avg_volatility_index,total_tier_market_cap,highly_volatile_count
Large Cap (>$10B),11,6257.13,2.54,2.086653143757E12,0
Mid Cap ($1B - $10B),39,247.41,2.98,1.52884416781E11,1


In [0]:
SELECT DISTINCT market_cap_tier, COUNT(*) as coin_count 
FROM crypto_gold_volatility 
GROUP BY market_cap_tier;

market_cap_tier,coin_count
Large Cap (>$10B),11
Mid Cap ($1B - $10B),39


Databricks visualization. Run in Databricks to view.

In [0]:
-- ==============================================================================
-- ANALYSIS 1: MARKET SENTIMENT & BREADTH (BULL VS. BEAR)
-- Business Question: Is the overall market trending up or down today?
-- Why this matters: Instead of just looking at Bitcoin, we want to know 
-- the momentum of the top 50 coins combined.
-- ==============================================================================

SELECT 
    COUNT(coin_id) AS total_coins,
    SUM(CASE WHEN price_change_percentage_24h > 0 THEN 1 ELSE 0 END) AS coins_in_green,
    SUM(CASE WHEN price_change_percentage_24h < 0 THEN 1 ELSE 0 END) AS coins_in_red,
    ROUND(AVG(price_change_percentage_24h), 2) AS average_market_return_pct
FROM crypto_gold_volatility;

total_coins,coins_in_green,coins_in_red,average_market_return_pct
50,42,7,1.57


In [0]:
-- ==============================================================================
-- ANALYSIS 2: RISK VS. REWARD QUADRANT
-- Business Question: Are the highly volatile coins actually making money, or 
-- are they just crashing? 
-- Why this matters: High volatility is only good if the price is going UP. 
-- We categorize coins into risk/reward buckets.
-- ==============================================================================

SELECT 
    symbol,
    name,
    current_price,
    price_change_percentage_24h,
    volatility_index,
    CASE 
        WHEN is_highly_volatile = true AND price_change_percentage_24h > 0 THEN 'High Risk / High Reward'
        WHEN is_highly_volatile = true AND price_change_percentage_24h <= 0 THEN 'High Risk / Heavy Loss'
        WHEN is_highly_volatile = false AND price_change_percentage_24h > 0 THEN 'Stable Growth'
        ELSE 'Stable Decline'
    END AS investment_profile
FROM crypto_gold_volatility
ORDER BY price_change_percentage_24h DESC;

symbol,name,current_price,price_change_percentage_24h,volatility_index,investment_profile
ONDO,Ondo,0.398146,14.11002,15.73,High Risk / High Reward
ADA,Cardano,0.174227,6.86239,8.49,Stable Growth
UNI,Uniswap,3.69,6.7435,7.59,Stable Growth
BCH,Bitcoin Cash,224.37,5.59559,5.7,Stable Growth
HBAR,Hedera,0.068306,3.418,3.66,Stable Growth
HYPE,Hyperliquid,62.74,3.40951,4.34,Stable Growth
LINK,Chainlink,8.69,3.33634,4.72,Stable Growth
XRP,XRP,1.13,3.254,4.6,Stable Growth
NEAR,NEAR Protocol,1.99,2.99604,6.53,Stable Growth
TAO,Bittensor,199.91,2.77011,4.06,Stable Growth


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
-- ==============================================================================
-- ANALYSIS 3: OUTLIER DETECTION USING PERCENTILES (WINDOW FUNCTION)
-- Business Question: Which specific coins are in the top 10% most volatile 
-- of the entire market?
-- Why this matters: PERCENT_RANK() allows us to dynamically find the top 10% 
-- without hardcoding a specific volatility number. 
-- ==============================================================================

WITH RankedVolatility AS (
    SELECT 
        symbol,
        name,
        market_cap_tier,
        volatility_index,
        -- PERCENT_RANK calculates the relative rank of a row as a decimal between 0 and 1
        ROUND(PERCENT_RANK() OVER (ORDER BY volatility_index ASC), 2) AS volatility_percentile
    FROM crypto_gold_volatility
)
SELECT * 
FROM RankedVolatility
-- Filter for coins in the 90th percentile (top 10% most volatile)
WHERE volatility_percentile >= 0.90
ORDER BY volatility_percentile DESC;

symbol,name,market_cap_tier,volatility_index,volatility_percentile
ONDO,Ondo,Mid Cap ($1B - $10B),15.73,1.0
ADA,Cardano,Mid Cap ($1B - $10B),8.49,0.98
UNI,Uniswap,Mid Cap ($1B - $10B),7.59,0.96
NEAR,NEAR Protocol,Mid Cap ($1B - $10B),6.53,0.94
M,MemeCore,Mid Cap ($1B - $10B),5.98,0.92
BCH,Bitcoin Cash,Mid Cap ($1B - $10B),5.7,0.9


In [0]:
-- ==============================================================================
-- ANALYSIS 4: THE "HIDDEN GEMS" (HIGH DOMINANCE, LOW PRICE)
-- Business Question: Are there any coins holding a massive share of the market 
-- (high dominance) but still trading at a relatively low price per coin?
-- Why this matters: Retail investors often look for "cheap" coins that already have strong market backing.
-- ==============================================================================

SELECT 
    symbol,
    name,
    current_price,
    market_dominance_pct,
    volatility_index
FROM crypto_gold_volatility
-- Filter for coins that hold at least 1% of the entire top 50 market cap
WHERE market_dominance_pct >= 1.00 
-- But are still trading for under $10
  AND current_price < 10.00
ORDER BY market_dominance_pct DESC;

symbol,name,current_price,market_dominance_pct,volatility_index
USDT,Tether,0.999188,8.22,0.05
USDC,USDC,0.999847,3.27,0.03
XRP,XRP,1.13,3.15,4.6
TRX,TRON,0.327386,1.39,0.7


In [0]:
SELECT symbol, current_price, last_updated_timestamp FROM crypto_gold_volatility LIMIT 5;

symbol,current_price,last_updated_timestamp
HYPE,60.19,2026-07-21T19:37:22.980Z
XRP,1.16,2026-07-21T19:37:33.967Z
ETH,1922.66,2026-07-21T19:37:34.065Z
BTC,66395.0,2026-07-21T19:37:30.786Z
DOGE,0.073523,2026-07-21T19:37:34.755Z
